In [ ]:
import os
import glob
import re
import csv
import cv2
import time
import numpy as np

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


def guardar_curva_distancia(filas, ruta_png, titulo):
    # Distancia P1-P2 (px) con eje derecho reescalado a deformacion (%). Colores aptos
    # para deuteranopia (azul / naranja), nunca verde ni amarillo.
    if not filas:
        return
    frames = [fila[0] for fila in filas]
    distancias = [fila[6] for fila in filas]
    d0 = distancias[0] if distancias[0] else 1.0

    fig, ax_izq = plt.subplots(figsize=(9, 5))
    ax_izq.plot(frames, distancias, color="#1f77b4", linewidth=2)
    ax_izq.set_xlabel("Frame")
    ax_izq.set_ylabel("Distancia P1-P2 (px)", color="#1f77b4")
    ax_izq.tick_params(axis="y", labelcolor="#1f77b4")
    ax_izq.grid(True, alpha=0.3)

    ax_der = ax_izq.twinx()
    lo, hi = ax_izq.get_ylim()
    ax_der.set_ylim((lo - d0) / d0 * 100.0, (hi - d0) / d0 * 100.0)
    ax_der.set_ylabel("Deformacion respecto al frame 0 (%)", color="#ff7f0e")
    ax_der.tick_params(axis="y", labelcolor="#ff7f0e")

    ax_izq.set_title(titulo)
    fig.tight_layout()
    fig.savefig(ruta_png, dpi=150, bbox_inches="tight")
    plt.close(fig)

In [ ]:
carpeta_imagenes_raiz = r"datos"
carpeta_salida_raiz = r"resultados"

carpetas_a_procesar = []  # vacio = todas; p.ej. ["T3_02"] para una sola

os.makedirs(carpeta_salida_raiz, exist_ok=True)

tam_kernel_tophat = 41
percentil_umbral = 97.5
tam_kernel_cierre = 9

area_minima = 200
area_maxima = 20000

umbral_mediana_probeta = 45
fraccion_rechazo_y_global = 0.93

relacion_aspecto_min = 0.30
relacion_aspecto_max = 3.00
circularidad_minima = 0.05

alpha_ema = 0.30
tolerancia_area = 0.80
tolerancia_media_intensidad = 65.0

semi_lado_roi = 220
escala_reintento_roi = 1.8

guardar_debug_roi = True
guardar_roi_cada_n = 1

salto_distancia_abs = 35.0
salto_distancia_rel = 0.08
guardar_debug_saltos = True

fraccion_min_sep_y_frame0 = 0.20

umbral_fractura_px = 20.0     # salto minimo (px) para marcar fractura y NO suavizar a traves
factor_fractura_jitter = 6.0  # ademas el salto debe superar 6x el temblor tipico
ventana_suavizado = 5         # media movil por segmentos para estabilizar la posicion

In [ ]:
def clave_natural(path):
    nombre_archivo = os.path.basename(path)

    trozos = re.split(r"(\d+)", nombre_archivo)

    clave = []
    for t in trozos:
        if t.isdigit():
            clave.append(int(t))
        else:
            clave.append(t.lower())

    return clave

def a_uint8(imagen):
    if imagen.ndim == 3:
        num_canales = imagen.shape[2]

        if num_canales == 3:
            imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)
        elif num_canales == 4:
            imagen = cv2.cvtColor(imagen, cv2.COLOR_BGRA2GRAY)

    if imagen.dtype == np.uint8:
        return imagen

    g = imagen.astype(np.float32)

    g = g - g.min()
    maximo = g.max()

    if maximo > 0:
        g = g / maximo

    g = (g * 255.0).astype(np.uint8)
    return g

def estimar_rango_x_probeta(gris_u8, umbral_mediana=45):
    mediana_columnas = np.median(gris_u8, axis=0)
    columnas = np.where(mediana_columnas < umbral_mediana)[0]

    if len(columnas) < 10:
        ancho = gris_u8.shape[1]
        x_ini = int(0.25 * ancho)
        x_fin = int(0.85 * ancho)
        return x_ini, x_fin

    cortes = np.where(np.diff(columnas) > 1)[0] + 1
    grupos = np.split(columnas, cortes)

    mejor_grupo = max(grupos, key=len)

    x_ini = int(mejor_grupo[0])
    x_fin = int(mejor_grupo[-1])
    return x_ini, x_fin

def circularidad_componente(mascara_u8):
    contornos, _ = cv2.findContours(mascara_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contornos:
        return 0.0

    contorno = max(contornos, key=cv2.contourArea)

    area = float(cv2.contourArea(contorno))
    perimetro = float(cv2.arcLength(contorno, True))

    if perimetro <= 1e-6:
        return 0.0

    circularidad = 4.0 * np.pi * area / (perimetro * perimetro + 1e-6)
    return float(circularidad)

def caja_roi(centro_x, centro_y, semi_lado, ancho, alto):
    x_ini = centro_x - semi_lado
    y_ini = centro_y - semi_lado
    x_fin = centro_x + semi_lado
    y_fin = centro_y + semi_lado

    x_ini = min(ancho - 1, x_ini)
    x_ini = max(0, x_ini)

    y_ini = min(alto - 1, y_ini)
    y_ini = max(0, y_ini)

    x_fin = min(ancho, x_fin)
    x_fin = max(0, x_fin)

    y_fin = min(alto, y_fin)
    y_fin = max(0, y_fin)

    x_ini = int(x_ini)
    y_ini = int(y_ini)
    x_fin = int(x_fin)
    y_fin = int(y_fin)

    if x_fin <= x_ini:
        x_fin = min(ancho, x_ini + 1)

    if y_fin <= y_ini:
        y_fin = min(alto, y_ini + 1)

    return x_ini, y_ini, x_fin, y_fin

def detectar_candidatos(gris_u8, x_ini_probeta, x_fin_probeta, offset_x=0, offset_y=0, alto_global=None):
    alto_roi, ancho_roi = gris_u8.shape

    if alto_global is None:
        alto_global = alto_roi

    kernel_tophat = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (tam_kernel_tophat, tam_kernel_tophat))
    imagen_tophat = cv2.morphologyEx(gris_u8, cv2.MORPH_TOPHAT, kernel_tophat)

    valores_no_cero = imagen_tophat[imagen_tophat > 0]
    if valores_no_cero.size > 0:
        umbral = np.percentile(valores_no_cero, percentil_umbral)
    else:
        umbral = 25.0

    umbral = max(10.0, float(umbral))

    _, mascara_binaria = cv2.threshold(imagen_tophat, umbral, 255, cv2.THRESH_BINARY)

    kernel_cierre = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (tam_kernel_cierre, tam_kernel_cierre))
    mascara_binaria = cv2.morphologyEx(mascara_binaria, cv2.MORPH_CLOSE, kernel_cierre, iterations=2)
    mascara_binaria = cv2.morphologyEx(mascara_binaria, cv2.MORPH_OPEN, kernel_cierre, iterations=1)

    num_componentes, etiquetas, estadisticas, centroides = cv2.connectedComponentsWithStats(mascara_binaria, connectivity=8)

    candidatos = []
    for etiqueta in range(1, num_componentes):
        x_caja, y_caja, ancho_caja, alto_caja, area = estadisticas[etiqueta, :5]

        # centroide subpixel ponderado por brillo (mas estable que el binario)
        ys_comp, xs_comp = np.where(etiquetas == etiqueta)
        pesos = gris_u8[ys_comp, xs_comp].astype(np.float64)
        suma_pesos = float(pesos.sum())
        if suma_pesos > 0:
            centro_x_roi = float((xs_comp * pesos).sum() / suma_pesos)
            centro_y_roi = float((ys_comp * pesos).sum() / suma_pesos)
        else:
            centro_x_roi, centro_y_roi = centroides[etiqueta]

        centro_x = float(centro_x_roi) + float(offset_x)
        centro_y = float(centro_y_roi) + float(offset_y)

        if area < area_minima or area > area_maxima:
            continue

        if x_ini_probeta is not None and x_fin_probeta is not None:
            if not (float(x_ini_probeta) <= centro_x <= float(x_fin_probeta)):
                continue

        if centro_y > fraccion_rechazo_y_global * float(alto_global):
            continue

        if alto_caja > 0:
            relacion_aspecto = float(ancho_caja) / float(alto_caja)
        else:
            relacion_aspecto = 999.0

        if not (relacion_aspecto_min <= relacion_aspecto <= relacion_aspecto_max):
            continue

        mascara_comp = (etiquetas == etiqueta).astype(np.uint8)
        mascara_comp = mascara_comp * 255

        circ = circularidad_componente(mascara_comp)
        if circ < circularidad_minima:
            continue

        media_intensidad = float(pesos.mean()) if pesos.size > 0 else 0.0

        candidato = {"etiqueta": int(etiqueta), "area": int(area), "centro_x": float(centro_x), "centro_y": float(centro_y), "media_intensidad": float(media_intensidad)}
        candidatos.append(candidato)

    return imagen_tophat, mascara_binaria, candidatos

def actualizar_ema(valor_ref, valor_nuevo):
    if valor_ref is None:
        return float(valor_nuevo)

    valor_ref_float = float(valor_ref)
    valor_nuevo_float = float(valor_nuevo)

    parte_anterior = (1.0 - alpha_ema) * valor_ref_float
    parte_nueva = alpha_ema * valor_nuevo_float

    valor_filtrado = parte_anterior + parte_nueva
    return float(valor_filtrado)

def cumple_tolerancias(candidato, area_ref, intensidad_ref):
    if area_ref is not None:
        diferencia_area = abs(float(candidato["area"]) - float(area_ref))
        if diferencia_area > tolerancia_area * float(area_ref):
            return False

    if intensidad_ref is not None:
        diferencia_i = abs(float(candidato["media_intensidad"]) - float(intensidad_ref))
        if diferencia_i > tolerancia_media_intensidad:
            return False

    return True

def puntuar_candidato(candidato, objetivo_xy, area_ref, intensidad_ref):
    dx = float(candidato["centro_x"]) - float(objetivo_xy[0])
    dy = float(candidato["centro_y"]) - float(objetivo_xy[1])

    dist2 = dx * dx + dy * dy

    penal_area = 0.0
    if area_ref is not None:
        area_ref_float = float(area_ref)
        if area_ref_float > 1.0:
            diferencia_area = abs(float(candidato["area"]) - area_ref_float)
            penal_area = diferencia_area / area_ref_float

    penal_i = 0.0
    if intensidad_ref is not None:
        diferencia_i = abs(float(candidato["media_intensidad"]) - float(intensidad_ref))
        penal_i = diferencia_i / 255.0

    coste = dist2 + 5000.0 * penal_area + 1500.0 * penal_i
    return float(coste)

def elegir_mejor_candidato(candidatos, objetivo_xy, area_ref, intensidad_ref):
    if not candidatos:
        return None

    candidatos_validos = []
    for candidato in candidatos:
        if cumple_tolerancias(candidato, area_ref, intensidad_ref):
            candidatos_validos.append(candidato)

    if candidatos_validos:
        pool_candidatos = candidatos_validos
    else:
        pool_candidatos = candidatos

    mejor_candidato = None
    mejor_coste = 1e30

    for candidato in pool_candidatos:
        coste = puntuar_candidato(candidato, objetivo_xy, area_ref, intensidad_ref)

        if coste < mejor_coste:
            mejor_coste = coste
            mejor_candidato = candidato

    return mejor_candidato

def elegir_dos_marcadores_frame0(candidatos, alto):
    if len(candidatos) < 2:
        return None

    candidatos_ordenados = sorted(candidatos, key=lambda d: float(d["media_intensidad"]), reverse=True)

    candidatos_top = candidatos_ordenados[:12]

    separacion_min_y = fraccion_min_sep_y_frame0 * float(alto)

    mejor_pareja = None
    mejor_puntuacion = -1e30

    for i in range(len(candidatos_top)):
        for j in range(i + 1, len(candidatos_top)):
            cand_a = candidatos_top[i]
            cand_b = candidatos_top[j]

            separacion_y = abs(float(cand_a["centro_y"]) - float(cand_b["centro_y"]))
            if separacion_y < separacion_min_y:
                continue

            puntuacion = float(cand_a["media_intensidad"]) + float(cand_b["media_intensidad"])
            if puntuacion > mejor_puntuacion:
                mejor_puntuacion = puntuacion
                mejor_pareja = (cand_a, cand_b)

    if mejor_pareja is None:
        mejor_pareja = (candidatos_top[0], candidatos_top[1])

    cand_1, cand_2 = mejor_pareja

    if float(cand_1["centro_y"]) <= float(cand_2["centro_y"]):
        return cand_1, cand_2

    return cand_2, cand_1

def guardar_mosaico_debug_roi(roi_gris_u8, roi_tophat_u8, roi_binaria_u8, punto_xy, ruta_salida, texto_titulo=""):
    imagen_gris_bgr = cv2.cvtColor(roi_gris_u8, cv2.COLOR_GRAY2BGR)
    imagen_tophat_bgr = cv2.cvtColor(roi_tophat_u8, cv2.COLOR_GRAY2BGR)
    imagen_binaria_bgr = cv2.cvtColor(roi_binaria_u8, cv2.COLOR_GRAY2BGR)

    x_int = int(punto_xy[0])
    y_int = int(punto_xy[1])

    imagenes = [imagen_gris_bgr, imagen_tophat_bgr, imagen_binaria_bgr]
    for imagen_bgr in imagenes:
        cv2.circle(imagen_bgr, (x_int, y_int), 8, (0, 255, 0), 2)

    if texto_titulo:
        cv2.putText(imagen_gris_bgr, str(texto_titulo), (10, 35), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)

    mosaico = cv2.hconcat([imagen_gris_bgr, imagen_tophat_bgr, imagen_binaria_bgr])

    cv2.imwrite(ruta_salida, mosaico)

def dibujar_puntos(gris_u8, puntos, texto_titulo=None):
    imagen_bgr = cv2.cvtColor(gris_u8, cv2.COLOR_GRAY2BGR)

    (x1, y1), (x2, y2) = puntos

    x1_int = int(x1)
    y1_int = int(y1)
    x2_int = int(x2)
    y2_int = int(y2)

    cv2.circle(imagen_bgr, (x1_int, y1_int), 10, (0, 255, 0), 2)
    cv2.circle(imagen_bgr, (x2_int, y2_int), 10, (0, 255, 0), 2)

    cv2.putText(imagen_bgr, "P1", (x1_int + 12, y1_int - 12), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
    cv2.putText(imagen_bgr, "P2", (x2_int + 12, y2_int - 12), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)

    if texto_titulo:
        cv2.putText(imagen_bgr, str(texto_titulo), (25, 70), cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0, 255, 255), 3)

    return imagen_bgr

def distancia_entre_puntos(puntos):
    (x1, y1), (x2, y2) = puntos
    distancia = float(np.hypot(x2 - x1, y2 - y1))
    return distancia

def es_salto_distancia(distancia_px, distancia_previa):
    if distancia_previa is None:
        return False, 0.0, 0.0

    delta_abs = abs(float(distancia_px) - float(distancia_previa))
    delta_rel = delta_abs / max(float(distancia_previa), 1e-6)

    hay_salto = (delta_abs >= salto_distancia_abs) and (delta_rel >= salto_distancia_rel)
    return bool(hay_salto), float(delta_abs), float(delta_rel)

def procesar_carpeta(nombre_carpeta):
    carpeta_imagenes = os.path.join(carpeta_imagenes_raiz, nombre_carpeta)
    carpeta_salida = os.path.join(carpeta_salida_raiz, f"{nombre_carpeta}_out")
    os.makedirs(carpeta_salida, exist_ok=True)

    ruta_csv_salida = os.path.join(carpeta_salida, "distancias_CV_V3.csv")
    ruta_preview_salida = os.path.join(carpeta_salida, "tiff00_CV_V3.png")

    carpeta_roi_debug = os.path.join(carpeta_salida, "ROI_debug")
    carpeta_roi_p1 = os.path.join(carpeta_roi_debug, "P1")
    carpeta_roi_p2 = os.path.join(carpeta_roi_debug, "P2")

    if guardar_debug_roi:
        os.makedirs(carpeta_roi_p1, exist_ok=True)
        os.makedirs(carpeta_roi_p2, exist_ok=True)

    rutas_tif = glob.glob(os.path.join(carpeta_imagenes, "*.tif"))
    rutas_tiff = glob.glob(os.path.join(carpeta_imagenes, "*.tiff"))

    rutas_imagenes = rutas_tif + rutas_tiff
    rutas_imagenes = sorted(rutas_imagenes, key=clave_natural)

    if not rutas_imagenes:
        print(f"[{nombre_carpeta}] No hay .tif/.tiff -> se salta.")
        return

    print(f"\n[{nombre_carpeta}] Frames encontrados: {len(rutas_imagenes)}")
    print(f"[{nombre_carpeta}] Output -> {carpeta_salida}")

    ruta_primera = rutas_imagenes[0]
    gris0_raw = cv2.imread(ruta_primera, cv2.IMREAD_UNCHANGED)
    gris0 = a_uint8(gris0_raw)

    alto, ancho = gris0.shape

    x_ini_probeta, x_fin_probeta = estimar_rango_x_probeta(gris0, umbral_mediana=umbral_mediana_probeta)

    _, _, candidatos0 = detectar_candidatos(gris0, x_ini_probeta, x_fin_probeta, offset_x=0, offset_y=0, alto_global=alto)

    pareja0 = elegir_dos_marcadores_frame0(candidatos0, alto)
    if pareja0 is None:
        print(f"[{nombre_carpeta}] ERROR: no pude detectar 2 marcadores en frame 0.")
        return

    cand_1, cand_2 = pareja0

    p1 = (float(cand_1["centro_x"]), float(cand_1["centro_y"]))
    p2 = (float(cand_2["centro_x"]), float(cand_2["centro_y"]))
    puntos0 = [p1, p2]

    area_1_ref = float(cand_1["area"])
    intensidad_1_ref = float(cand_1["media_intensidad"])
    area_2_ref = float(cand_2["area"])
    intensidad_2_ref = float(cand_2["media_intensidad"])

    imagen_preview = dibujar_puntos(gris0, puntos0, texto_titulo="frame 0 (GLOBAL)")
    cv2.imwrite(ruta_preview_salida, imagen_preview)

    print(f"[{nombre_carpeta}] Preview guardada en: {ruta_preview_salida}")

    filas = []

    distancia_previa = distancia_entre_puntos(puntos0)
    nombre_archivo_0 = os.path.basename(ruta_primera)

    fila0 = [0, nombre_archivo_0, float(p1[0]), float(p1[1]), float(p2[0]), float(p2[1]), float(distancia_previa)]
    filas.append(fila0)

    gris_previo = gris0.copy()
    puntos_previos = puntos0

    for indice_frame in range(1, len(rutas_imagenes)):
        ruta_imagen = rutas_imagenes[indice_frame]

        gris_raw = cv2.imread(ruta_imagen, cv2.IMREAD_UNCHANGED)
        gris = a_uint8(gris_raw)

        alto, ancho = gris.shape

        x_ini_probeta, x_fin_probeta = estimar_rango_x_probeta(gris, umbral_mediana=umbral_mediana_probeta)

        (x1_prev, y1_prev), (x2_prev, y2_prev) = puntos_previos

        semi_lado_p1 = int(semi_lado_roi)
        x0_p1, y0_p1, x1_p1, y1_p1 = caja_roi(x1_prev, y1_prev, semi_lado_p1, ancho, alto)

        roi_gris_p1 = gris[y0_p1:y1_p1, x0_p1:x1_p1]

        roi_tophat_p1, roi_binaria_p1, candidatos_p1 = detectar_candidatos(roi_gris_p1, x_ini_probeta, x_fin_probeta, offset_x=x0_p1,offset_y=y0_p1, alto_global=alto)

        mejor_p1 = elegir_mejor_candidato(candidatos_p1, (x1_prev, y1_prev), area_1_ref, intensidad_1_ref)

        if mejor_p1 is None:
            semi_lado_p1_grande = int(float(semi_lado_roi) * float(escala_reintento_roi))

            x0b_p1, y0b_p1, x1b_p1, y1b_p1 = caja_roi( x1_prev, y1_prev, semi_lado_p1_grande, ancho, alto)

            roi_gris_p1b = gris[y0b_p1:y1b_p1, x0b_p1:x1b_p1]

            roi_tophat_p1b, roi_binaria_p1b, candidatos_p1b = detectar_candidatos( roi_gris_p1b, x_ini_probeta, x_fin_probeta, offset_x=x0b_p1, offset_y=y0b_p1, alto_global=alto)

            mejor_p1 = elegir_mejor_candidato(candidatos_p1b, (x1_prev, y1_prev), area_1_ref, intensidad_1_ref)

            x0_p1 = x0b_p1
            y0_p1 = y0b_p1
            roi_gris_p1 = roi_gris_p1b
            roi_tophat_p1 = roi_tophat_p1b
            roi_binaria_p1 = roi_binaria_p1b

        if mejor_p1 is None:
            p1_nuevo = (float(x1_prev), float(y1_prev))
        else:
            p1_nuevo = (float(mejor_p1["centro_x"]), float(mejor_p1["centro_y"]))
            area_1_ref = actualizar_ema(area_1_ref, mejor_p1["area"])
            intensidad_1_ref = actualizar_ema(intensidad_1_ref, mejor_p1["media_intensidad"])

        if guardar_debug_roi and (indice_frame % int(guardar_roi_cada_n) == 0):
            punto_local_p1 = (p1_nuevo[0] - float(x0_p1), p1_nuevo[1] - float(y0_p1))
            ruta_roi_p1 = os.path.join(carpeta_roi_p1, f"ROI_P1_{indice_frame:04d}.png")

            guardar_mosaico_debug_roi( roi_gris_p1, roi_tophat_p1, roi_binaria_p1, punto_local_p1, ruta_roi_p1, texto_titulo=f"P1 frame {indice_frame}")

        semi_lado_p2 = int(semi_lado_roi)
        x0_p2, y0_p2, x1_p2, y1_p2 = caja_roi(x2_prev, y2_prev, semi_lado_p2, ancho, alto)

        roi_gris_p2 = gris[y0_p2:y1_p2, x0_p2:x1_p2]

        roi_tophat_p2, roi_binaria_p2, candidatos_p2 = detectar_candidatos( roi_gris_p2, x_ini_probeta, x_fin_probeta, offset_x=x0_p2, offset_y=y0_p2,
            alto_global=alto)

        mejor_p2 = elegir_mejor_candidato(candidatos_p2, (x2_prev, y2_prev), area_2_ref, intensidad_2_ref)

        if mejor_p2 is None:
            semi_lado_p2_grande = int(float(semi_lado_roi) * float(escala_reintento_roi))

            x0b_p2, y0b_p2, x1b_p2, y1b_p2 = caja_roi(x2_prev, y2_prev, semi_lado_p2_grande,ancho, alto)

            roi_gris_p2b = gris[y0b_p2:y1b_p2, x0b_p2:x1b_p2]

            roi_tophat_p2b, roi_binaria_p2b, candidatos_p2b = detectar_candidatos(roi_gris_p2b, x_ini_probeta, x_fin_probeta, offset_x=x0b_p2,
                offset_y=y0b_p2, alto_global=alto)

            mejor_p2 = elegir_mejor_candidato(candidatos_p2b,(x2_prev, y2_prev),area_2_ref,intensidad_2_ref)

            x0_p2 = x0b_p2
            y0_p2 = y0b_p2
            roi_gris_p2 = roi_gris_p2b
            roi_tophat_p2 = roi_tophat_p2b
            roi_binaria_p2 = roi_binaria_p2b

        if mejor_p2 is None:
            p2_nuevo = (float(x2_prev), float(y2_prev))
        else:
            p2_nuevo = (float(mejor_p2["centro_x"]), float(mejor_p2["centro_y"]))
            area_2_ref = actualizar_ema(area_2_ref, mejor_p2["area"])
            intensidad_2_ref = actualizar_ema(intensidad_2_ref, mejor_p2["media_intensidad"])

        if guardar_debug_roi and (indice_frame % int(guardar_roi_cada_n) == 0):
            punto_local_p2 = (p2_nuevo[0] - float(x0_p2), p2_nuevo[1] - float(y0_p2))
            ruta_roi_p2 = os.path.join(carpeta_roi_p2, f"ROI_P2_{indice_frame:04d}.png")

            guardar_mosaico_debug_roi(roi_gris_p2,roi_tophat_p2,roi_binaria_p2,punto_local_p2,ruta_roi_p2,texto_titulo=f"P2 frame {indice_frame}")

        puntos = [p1_nuevo, p2_nuevo]
        distancia_px = distancia_entre_puntos(puntos)

        hay_salto, delta_abs, delta_rel = es_salto_distancia(distancia_px, distancia_previa)
        if hay_salto:
            print(f"[{nombre_carpeta}] SALTO en frame {indice_frame}: "f"dist={distancia_px:.2f}, prev={distancia_previa:.2f}, "f"Δ={delta_abs:.2f} ({delta_rel * 100:.1f}%)")

            if guardar_debug_saltos:
                ruta_debug_prev = os.path.join(carpeta_salida,f"JUMP_prev_{indice_frame - 1:04d}_V3.png")
                ruta_debug_curr = os.path.join(carpeta_salida,f"JUMP_curr_{indice_frame:04d}_V3.png")

                titulo_prev = f"frame {indice_frame - 1} dist={distancia_previa:.2f}px"
                titulo_curr = (f"frame {indice_frame} dist={distancia_px:.2f}px  "f"d={delta_abs:.2f}px ({delta_rel * 100:.1f}%)")

                imagen_prev = dibujar_puntos(gris_previo, puntos_previos, texto_titulo=titulo_prev)
                imagen_curr = dibujar_puntos(gris, puntos, texto_titulo=titulo_curr)

                cv2.imwrite(ruta_debug_prev, imagen_prev)
                cv2.imwrite(ruta_debug_curr, imagen_curr)

        nombre_archivo = os.path.basename(ruta_imagen)

        fila = [int(indice_frame),nombre_archivo,float(p1_nuevo[0]),float(p1_nuevo[1]),float(p2_nuevo[0]),float(p2_nuevo[1]),float(distancia_px)]
        filas.append(fila)

        gris_previo = gris.copy()
        puntos_previos = puntos
        distancia_previa = distancia_px

    # --- Estabilizacion: centroide subpixel (en la deteccion) + suavizado de posicion ---
    # Se preserva el salto de fractura: no se suaviza a traves del mayor salto real.
    xs1 = [fila[2] for fila in filas]
    ys1 = [fila[3] for fila in filas]
    xs2 = [fila[4] for fila in filas]
    ys2 = [fila[5] for fila in filas]
    dist_bruta = [fila[6] for fila in filas]
    n_frames = len(filas)

    deltas = [abs(dist_bruta[i] - dist_bruta[i - 1]) for i in range(1, n_frames)]
    temblor = float(np.median(deltas)) if deltas else 0.0
    frame_fractura = None
    if deltas:
        i_max = int(np.argmax(deltas)) + 1
        salto_max = deltas[i_max - 1]
        if salto_max >= umbral_fractura_px and salto_max >= factor_fractura_jitter * max(temblor, 1e-6):
            frame_fractura = i_max

    def suavizar_segmento(valores):
        m = len(valores)
        semis = ventana_suavizado // 2
        suave = []
        for i in range(m):
            ini = max(0, i - semis)
            fin = min(m, i + semis + 1)
            suave.append(sum(valores[ini:fin]) / (fin - ini))
        return suave

    def suavizar(valores):
        if frame_fractura is None:
            return suavizar_segmento(valores)
        return suavizar_segmento(valores[:frame_fractura]) + suavizar_segmento(valores[frame_fractura:])

    xs1 = suavizar(xs1); ys1 = suavizar(ys1)
    xs2 = suavizar(xs2); ys2 = suavizar(ys2)

    filas_estables = []
    for i in range(n_frames):
        d = float(np.hypot(xs2[i] - xs1[i], ys2[i] - ys1[i]))
        filas_estables.append([filas[i][0], filas[i][1], xs1[i], ys1[i], xs2[i], ys2[i], d])
    filas = filas_estables

    # --- CSV (posiciones y distancia estabilizadas) ---
    with open(ruta_csv_salida, "w", newline="", encoding="utf-8") as f:
        escritor_csv = csv.writer(f)
        escritor_csv.writerow(["frame", "file", "x1", "y1", "x2", "y2", "dist_px"])
        escritor_csv.writerows(filas)
    print(f"[{nombre_carpeta}] CSV guardado en: {ruta_csv_salida}")
    if frame_fractura is not None:
        print(f"[{nombre_carpeta}] Fractura en frame {frame_fractura} (salto {deltas[frame_fractura - 1]:.1f} px)")
    else:
        print(f"[{nombre_carpeta}] Sin fractura detectada (ensayo suave)")

    # --- Curva de distancia / deformacion (estabilizada) ---
    ruta_curva = os.path.join(carpeta_salida, "curva_distancia_V3.png")
    guardar_curva_distancia(filas, ruta_curva, f"{nombre_carpeta} - distancia P1-P2")
    print(f"[{nombre_carpeta}] Curva guardada en: {ruta_curva}")

    # --- Carpeta 'puntos_unidos': P1 y P2 unidos por linea, con distancia y frame ---
    carpeta_puntos = os.path.join(carpeta_salida, "puntos_unidos")
    os.makedirs(carpeta_puntos, exist_ok=True)
    for i, ruta_imagen in enumerate(rutas_imagenes):
        gris_i = a_uint8(cv2.imread(ruta_imagen, cv2.IMREAD_UNCHANGED))
        lienzo = cv2.cvtColor(gris_i, cv2.COLOR_GRAY2BGR)
        punto1 = (int(round(filas[i][2])), int(round(filas[i][3])))
        punto2 = (int(round(filas[i][4])), int(round(filas[i][5])))
        cv2.line(lienzo, punto1, punto2, (255, 255, 255), 2)
        cv2.circle(lienzo, punto1, 10, (0, 128, 255), 2)
        cv2.circle(lienzo, punto2, 10, (0, 128, 255), 2)
        cv2.putText(lienzo, "P1", (punto1[0] + 14, punto1[1]), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 128, 255), 2)
        cv2.putText(lienzo, "P2", (punto2[0] + 14, punto2[1]), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 128, 255), 2)
        cv2.putText(lienzo, f"frame {int(filas[i][0])}  dist = {filas[i][6]:.1f} px", (20, 45),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 255), 2)
        cv2.imwrite(os.path.join(carpeta_puntos, f"puntos_{int(filas[i][0]):04d}.png"), lienzo)
    print(f"[{nombre_carpeta}] Puntos unidos -> {carpeta_puntos}")

In [ ]:
carpetas = os.listdir(carpeta_imagenes_raiz)

nombres_carpetas = []
for nombre in carpetas:
    ruta = os.path.join(carpeta_imagenes_raiz, nombre)

    es_carpeta = os.path.isdir(ruta)
    if es_carpeta:
        nombres_carpetas.append(nombre)

nombres_carpetas = sorted(nombres_carpetas, key=clave_natural)

print("Carpetas a procesar:", nombres_carpetas)

for nombre_carpeta in nombres_carpetas:
    if carpetas_a_procesar and nombre_carpeta not in carpetas_a_procesar:
        continue
    try:
        t0 = time.perf_counter()

        procesar_carpeta(nombre_carpeta)

        t1 = time.perf_counter()
        segundos = t1 - t0

        print(f"[{nombre_carpeta}] Tiempo: {segundos:.1f} s ({segundos/60:.2f} min)")

    except Exception as e:
        print(f"[{nombre_carpeta}] EXCEPCION: {e}")

print("FIN.")